# 02. Approach 1: Classical Computer Vision & Machine Learning
### Avocado Ripening Stage Detection & Classification System
**Pipeline:** CIELAB ($L, a^*$) + HSV ($H, S$) + Uniform LBP Texture $\rightarrow$ StandardScaler $\rightarrow$ Support Vector Machine (RBF Kernel)

---
## Objective
Implement an interpretable, domain-informed classical computer vision pipeline that explicitly captures the biological visual markers of avocado ripening:
1. **Color Dynamics:** Green-to-purple/black skin color transition via CIELAB and HSV color histograms.
2. **Surface Texture:** Skin pebbled texture variations via Local Binary Patterns (LBP).
3. **Classification:** Calibrated Support Vector Classifier with balanced class weighting.


In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path('.').resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    CLASSES,
    CLASS_TO_IDX,
    SPLIT_MANIFEST_PATH,
    CLASSICAL_MODEL_PATH,
    EVALUATION_OUTPUTS_DIR,
)
from src.preprocessing import load_split_manifest
from src.classical_model import (
    extract_classical_features,
    build_classical_pipeline,
    save_classical_model,
    load_classical_model,
)
from src.evaluation import evaluate_classical

print(f"Classes: {CLASSES}")


## 1. Feature Engineering: Biological Color & Texture Signals
The feature extraction extracts 74 dimensions:
- **CIELAB Color Space (32 bins):** 16 bins for $L$ (luminance) + 16 bins for $a^*$ (green-to-red axis).
- **HSV Color Space (32 bins):** 16 bins for Hue + 16 bins for Saturation.
- **Local Binary Patterns (10 bins):** Uniform LBP ($P=8, R=1$) capturing micro-texture coarseness.


In [ ]:
# Load split manifest
manifest_df = load_split_manifest(SPLIT_MANIFEST_PATH)
train_df = manifest_df[manifest_df['split'] == 'train'].reset_index(drop=True)
val_df = manifest_df[manifest_df['split'] == 'val'].reset_index(drop=True)
test_df = manifest_df[manifest_df['split'] == 'test'].reset_index(drop=True)

print(f"Partitions: Train={len(train_df)}, Val={len(val_df)}, Test={len(test_df)}")


## 2. Feature Extraction Across Training Set
Extracting normalized color and texture feature vectors for all training images.


In [ ]:
X_train, y_train = [], []
for _, row in train_df.iterrows():
    try:
        feats = extract_classical_features(row['file_path'])
        X_train.append(feats)
        y_train.append(row['class_idx'])
    except Exception as e:
        pass

X_train = np.array(X_train)
y_train = np.array(y_train)
print(f"Training feature matrix shape: {X_train.shape}")


## 3. Training Support Vector Classifier
Fitting `StandardScaler` followed by `SVC(kernel='rbf', probability=True, class_weight='balanced')`.


In [ ]:
pipeline = build_classical_pipeline(c_param=2.0)
pipeline.fit(X_train, y_train)
save_classical_model(pipeline, save_path=CLASSICAL_MODEL_PATH)
print("Classical pipeline fitted and saved successfully.")


## 4. Evaluation on Held-Out Test Set
Evaluating the classical model on the 15% held-out test partition.


In [ ]:
report, cm, latency_ms = evaluate_classical(test_df, model_path=CLASSICAL_MODEL_PATH)
print(f"Test Accuracy: {report['accuracy']*100:.2f}%")
print(f"Macro F1-Score: {report['macro avg']['f1-score']:.4f}")
print(f"Inference Latency: {latency_ms:.2f} ms")


## Conclusion
- Approach 1 demonstrates strong performance and extremely fast inference.
- Handcrafted features provide domain interpretability.
- Proceed to `03_approach_2_deep_learning.ipynb` to evaluate hierarchical transfer learning with MobileNetV2.
